In [1]:
suppressPackageStartupMessages({
  library(tradeSeq)
  library(dplyr)
  library(tidyr)
  library(tibble)
  library(ggplot2)
  library(matrixStats)
  library(cluster)
  library(RColorBrewer)
  library(SingleCellExperiment)
  library(slingshot)
  library(Matrix)
  library(BiocParallel)
  library(patchwork)
})

RNGversion("4.3.0")
set.seed(1)

# This notebook is intentionally self-contained with respect to the older
# DimensionalityReductionClustering notebooks. It reads primary tradeSeq inputs
# and writes regenerated outputs to SELF_CONTAINED_OUTDIR.
workspace_notebook_dir <- normalizePath(getwd(), mustWork = TRUE)
original_path_file <- file.path(workspace_notebook_dir, "OriginalDirectoryPath.txt")
if (!file.exists(original_path_file)) {
  stop("Run this notebook from k562-tnfa-puromycin.hg38/, where OriginalDirectoryPath.txt is available.")
}

puro_notebook_dir <- normalizePath(path.expand(readLines(original_path_file, warn = FALSE)[1]), mustWork = TRUE)
puro_root <- normalizePath(file.path(puro_notebook_dir, ".."), mustWork = TRUE)
analysis_root <- normalizePath(file.path(puro_root, ".."), mustWork = TRUE)
root_k562a <- file.path(analysis_root, "k562-tnfa.hg38.rep")
root_lps <- file.path(analysis_root, "thp1-k562-lps.hg38.rep")
root_puro <- puro_root

SELF_CONTAINED_OUTDIR <- Sys.getenv(
  "CHRONOPACK_DIMRED_OUTDIR",
  file.path(workspace_notebook_dir, "self_contained_outputs", "global_analysis_extended")
)
dir.create(SELF_CONTAINED_OUTDIR, recursive = TRUE, showWarnings = FALSE)
SELF_CONTAINED_OUTDIR <- normalizePath(SELF_CONTAINED_OUTDIR, mustWork = TRUE)

# Keep subsequent saves local to the regenerated output directory.
setwd(SELF_CONTAINED_OUTDIR)
SELF_CONTAINED_OUTDIR


[1] "/new-stg/home/alex/tmp/ChronoPack/k562-tnfa-puromycin.hg38/self_contained_outputs/global_analysis_extended"

In [2]:
get_descriptors <- function(
  traj_mat, tt,
  pseudocount = 1e-5,
  N_minutes = 10,
  z_diff_threshold = 0.10,
  require_consecutive = 3,
  onset_mode = c("early_lm", "horizontal"),
  owp_onset_fraction = 0.05
) {
  onset_mode <- match.arg(onset_mode)

  trapz_simple <- function(x, y) {
    if (length(x) < 2) return(0)
    sum(diff(x) * (y[-1] + y[-length(y)]) / 2)
  }

  # OWP = positive area / area under running-max envelope.
  # Expression: subtract initial value first.
  # Derivative: zero is the natural baseline.
  compute_owp <- function(y, t, onset_fraction = 0.05, baseline = c("first", "zero")) {
    baseline <- match.arg(baseline)
    keep <- is.finite(y) & is.finite(t)
    y <- y[keep]; t <- t[keep]
    if (length(y) < 2) return(NA_real_)

    if (baseline == "first") y <- y - y[1]
    yp <- pmax(y, 0)

    ymax <- max(yp, na.rm = TRUE)
    if (!is.finite(ymax) || ymax <= 0) return(NA_real_)

    onset <- which(yp >= onset_fraction * ymax)[1]
    if (is.na(onset) || onset >= length(yp)) return(NA_real_)

    yp <- yp[onset:length(yp)]
    t  <- t[onset:length(t)]
    env <- cummax(yp)

    denom <- trapz_simple(t, env)
    if (!is.finite(denom) || denom <= 0) return(NA_real_)

    trapz_simple(t, yp) / denom
  }

  calc_shape_metrics <- function(y, tt) {
    if (length(tt) < 3) return(list(mean_curv = NA_real_, max_curv = NA_real_, mean_turn = NA_real_, max_turn = NA_real_))

    tmin <- min(tt); tmax <- max(tt)
    if (!is.finite(tmin) || !is.finite(tmax) || tmax <= tmin)
      return(list(mean_curv = NA_real_, max_curv = NA_real_, mean_turn = NA_real_, max_turn = NA_real_))

    x <- (tt - tmin) / (tmax - tmin)
    ymin <- min(y, na.rm = TRUE); ymax <- max(y, na.rm = TRUE)
    if (!is.finite(ymin) || !is.finite(ymax) || ymax <= ymin)
      return(list(mean_curv = 0, max_curv = 0, mean_turn = 0, max_turn = 0))

    z <- (y - ymin) / (ymax - ymin)
    n <- length(x)

    dx_l <- x[2:(n - 1)] - x[1:(n - 2)]
    dx_r <- x[3:n] - x[2:(n - 1)]
    slope_l <- (z[2:(n - 1)] - z[1:(n - 2)]) / dx_l
    slope_r <- (z[3:n] - z[2:(n - 1)]) / dx_r
    dz  <- (z[3:n] - z[1:(n - 2)]) / (x[3:n] - x[1:(n - 2)])
    d2z <- 2 * (slope_r - slope_l) / (dx_l + dx_r)
    kappa <- abs(d2z) / (1 + dz^2)^(3/2)

    u <- cbind(x[2:(n - 1)] - x[1:(n - 2)], z[2:(n - 1)] - z[1:(n - 2)])
    v <- cbind(x[3:n] - x[2:(n - 1)], z[3:n] - z[2:(n - 1)])
    denom <- sqrt(rowSums(u^2)) * sqrt(rowSums(v^2))

    cosang <- rep(1, length(denom))
    ok <- is.finite(denom) & denom > 0
    cosang[ok] <- rowSums(u * v)[ok] / denom[ok]
    turn_angle <- acos(pmax(-1, pmin(1, cosang))) / pi

    list(
      mean_curv = mean(kappa, na.rm = TRUE),
      max_curv = max(kappa, na.rm = TRUE),
      mean_turn = mean(turn_angle, na.rm = TRUE),
      max_turn = max(turn_angle, na.rm = TRUE)
    )
  }

  calc_onset_zdiff <- function(y, tt, N_minutes = 10, z_diff_threshold = 0.25,
                               require_consecutive = 1, mode = c("early_lm", "horizontal")) {
    mode <- match.arg(mode)
    keep <- is.finite(y) & is.finite(tt)
    y <- y[keep]; tt <- tt[keep]
    if (length(y) < 2) return(NA_real_)

    y_sd <- sd(y, na.rm = TRUE)
    if (!is.finite(y_sd) || y_sd == 0) y_sd <- 1
    df <- data.frame(time = tt, expr_z = (y - mean(y, na.rm = TRUE)) / y_sd)
    early_df <- df[df$time <= N_minutes, , drop = FALSE]
    if (nrow(early_df) < 2) return(NA_real_)

    if (mode == "early_lm") {
      fit <- lm(expr_z ~ time, data = early_df)
      df$blue_fit <- predict(fit, newdata = df)
    } else {
      df$blue_fit <- mean(early_df$expr_z, na.rm = TRUE)
    }

    above <- abs(df$expr_z - df$blue_fit) > z_diff_threshold
    if (require_consecutive <= 1) {
      onset_idx <- which(above)[1]
    } else {
      r <- rle(above)
      ends <- cumsum(r$lengths); starts <- ends - r$lengths + 1
      good <- which(r$values & r$lengths >= require_consecutive)
      onset_idx <- if (length(good)) starts[good[1]] else NA_integer_
    }

    if (is.na(onset_idx)) NA_real_ else df$time[onset_idx]
  }

  # ----------------------------------------------------------------
  # Trajectory + derivatives
  # ----------------------------------------------------------------

  baseline <- traj_mat[, 1]
  traj0 <- sweep(traj_mat, 1, baseline, "-")

  dt <- diff(tt)
  if (any(dt <= 0)) stop("Non-increasing time grid in tt.")

  dtraj <- sweep(
    traj_mat[, -1, drop = FALSE] - traj_mat[, -ncol(traj_mat), drop = FALSE],
    2, dt, "/"
  )
  t_dtraj <- (tt[-1] + tt[-length(tt)]) / 2

  ddt <- diff(t_dtraj)
  ddtraj <- sweep(
    dtraj[, -1, drop = FALSE] - dtraj[, -ncol(dtraj), drop = FALSE],
    2, ddt, "/"
  )

  # ----------------------------------------------------------------
  # Peak descriptors
  # ----------------------------------------------------------------

    peak_idx <- max.col(abs(traj0), ties.method = "first")
    t_peak <- tt[peak_idx]
    y_peak <- traj_mat[cbind(seq_len(nrow(traj_mat)), peak_idx)]
    
    peak_dexp <- expm1(y_peak) - expm1(baseline)
    
    peak_log2fc <- log2(
      (expm1(y_peak) + pseudocount) /
      (expm1(baseline) + pseudocount)
    )

  # ----------------------------------------------------------------
  # Onset + AUC + OWP
  # ----------------------------------------------------------------

  t_onset <- vapply(seq_len(nrow(traj_mat)), function(i)
    calc_onset_zdiff(
      as.numeric(traj_mat[i, ]), tt,
      N_minutes, z_diff_threshold, require_consecutive, onset_mode
    ), numeric(1))

  auc_norm <- vapply(seq_len(nrow(traj_mat)), function(i) {
    y <- as.numeric(traj_mat[i, ])
    ymin <- min(y, na.rm = TRUE); ymax <- max(y, na.rm = TRUE)
    if (!is.finite(ymin) || !is.finite(ymax) || ymax <= ymin) return(0)
    trapz_simple(tt, (y - ymin) / (ymax - ymin)) / diff(range(tt))
  }, numeric(1))

  owp <- vapply(seq_len(nrow(traj_mat)), function(i)
    compute_owp(
      as.numeric(traj_mat[i, ]), tt,
      onset_fraction = owp_onset_fraction,
      baseline = "first"
    ), numeric(1))

  owp_ddt <- vapply(seq_len(nrow(dtraj)), function(i)
    compute_owp(
      as.numeric(dtraj[i, ]), t_dtraj,
      onset_fraction = owp_onset_fraction,
      baseline = "zero"
    ), numeric(1))

  # ----------------------------------------------------------------
  # Shape descriptors
  # ----------------------------------------------------------------

  shape_metrics <- lapply(seq_len(nrow(traj_mat)), function(i)
    calc_shape_metrics(as.numeric(traj_mat[i, ]), tt))

  mean_curv <- vapply(shape_metrics, `[[`, numeric(1), "mean_curv")
  max_curv  <- vapply(shape_metrics, `[[`, numeric(1), "max_curv")
  mean_turn <- vapply(shape_metrics, `[[`, numeric(1), "mean_turn")
  max_turn  <- vapply(shape_metrics, `[[`, numeric(1), "max_turn")

  data.frame(
    gene_celltype = rownames(traj_mat),
    t_onset = t_onset,
    t_peak = t_peak,
    baseline_exp = baseline,
    peak_exp = y_peak,
    peak_dlogexp = y_peak - baseline,
    peak_dexp = peak_dexp,              # NEW
    peak_log2fc = peak_log2fc,
    auc_norm = auc_norm,
    owp = owp,
    owp_ddt = owp_ddt,
    mean_curv = mean_curv,
    max_curv = max_curv,
    mean_turn = mean_turn,
    max_turn = max_turn,
    stringsAsFactors = FALSE
  )
}

In [3]:
# Load saved intermediates for targeted downstream refresh.
traj_mat <- readRDS(file.path(SELF_CONTAINED_OUTDIR, "traj_mat.rds"))
tt <- readRDS(file.path(SELF_CONTAINED_OUTDIR, "tt.rds"))
asso_df.ls <- readRDS(file.path(root_puro, "intmd", "global_analysis", "extended", "asso_df_ls.rds"))
scale_map_use <- c(K562a = 1, K562_puro = 1, K562b = 1, THP1 = 0.65)


In [4]:
set.seed(1)

z_row <- function(M) {
  mu <- rowMeans(M, na.rm = TRUE)
  sd <- matrixStats::rowSds(M, na.rm = TRUE)
  sd[!is.finite(sd) | sd == 0] <- 1
  sweep(sweep(M, 1, mu, "-"), 1, sd, "/")
}

sil_to_alpha <- function(s) {
  s <- pmax(-0.2, pmin(0.8, s))
  0.15 + (s + 0.2) / (0.8 + 0.2) * (1.0 - 0.15)
}

desc = get_descriptors(traj_mat, tt, N_minutes = 10/length(tt), z_diff_threshold = 0.2, require_consecutive = 30, onset_mode = 'horizontal', pseudocount=0)
gene_celltype <- rownames(traj_mat)
descriptor_key <- sub("^[^_]+_", "", gene_celltype)
descriptor_scale <- length(tt) / as.numeric(scale_map_use[descriptor_key])
descriptor_scale[!is.finite(descriptor_scale)] <- length(tt)
desc$t_onset <- desc$t_onset * descriptor_scale
desc$t_peak <- desc$t_peak * descriptor_scale


In [5]:
pc_df <- data.frame(gene_celltype = gene_celltype,
                    gene = sapply(gene_celltype, function(x) { strsplit(x, '_')[[1]][1] }, USE.NAMES = F),
                    cell_type = sub("^[^_]+_", "", gene_celltype)) %>% left_join(desc, by = c('gene_celltype'))
rownames(pc_df) = pc_df$gene_celltype
pc_df = pc_df %>% select(-gene_celltype)

for (nm in c("waldStat_1", "meanLogFC", "pvalue_1", "qval_1")) {
    pc_df[[nm]] <- NA_real_
}

# Build lookup table from asso_df.ls
lookup_df <- do.call(rbind, lapply(names(asso_df.ls), function(ct) {
                                df <- asso_df.ls[[ct]]
                                if (is.null(df) || nrow(df) == 0) return(NULL)
                            
                                out <- data.frame(cell_type = ct, gene = rownames(df), stringsAsFactors = FALSE, row.names = NULL)
                            
                                for (nm in c("waldStat_1", "meanLogFC", "pvalue_1", "qval_1")) {
                                  out[[paste0(nm, "_lookup")]] <- if (nm %in% colnames(df)) df[[nm]] else NA_real_
                                }
                        
                                out
                              })
                    )

pc_df <- pc_df %>% rownames_to_column("row_id") %>% left_join(lookup_df, by = c("cell_type", "gene")) %>%
                   mutate(
                     waldStat_1 = ifelse(is.na(waldStat_1), waldStat_1_lookup, waldStat_1),
                     meanLogFC  = ifelse(is.na(meanLogFC),  meanLogFC_lookup,  meanLogFC),
                     pvalue_1   = ifelse(is.na(pvalue_1),   pvalue_1_lookup,   pvalue_1),
                     qval_1     = ifelse(is.na(qval_1),     qval_1_lookup,     qval_1)) %>%
                   select(-waldStat_1_lookup, -meanLogFC_lookup, -pvalue_1_lookup, -qval_1_lookup) %>% column_to_rownames("row_id")

# Quick sanity check
cat("Remaining NA counts:\n")
print(colSums(is.na(pc_df[, c("waldStat_1", "meanLogFC", "pvalue_1", "qval_1")])))
print(head(pc_df[, c("gene", "waldStat_1", "meanLogFC", "pvalue_1", "qval_1")], 10))


Remaining NA counts:


waldStat_1  meanLogFC   pvalue_1     qval_1 
         0          0          0          0 


               gene waldStat_1 meanLogFC pvalue_1 qval_1
NFKBIA_K562a NFKBIA  1967.3776 2.8062932        0      0
CXCL3_K562a   CXCL3   742.3239 2.7863760        0      0
CD69_K562a     CD69   602.3647 3.4546323        0      0
CD83_K562a     CD83   566.9987 2.1289979        0      0
ARL4A_K562a   ARL4A   466.5694 0.3653517        0      0
ICAM1_K562a   ICAM1   439.2142 2.9057450        0      0
CXCL8_K562a   CXCL8   426.5580 3.1128228        0      0
BTG2_K562a     BTG2   351.1097 1.7552631        0      0
BIRC3_K562a   BIRC3   322.0169 3.6008999        0      0
DDIT4_K562a   DDIT4   302.2627 1.3388896        0      0


In [6]:
X <- z_row(traj_mat)

dt <- diff(tt)
Xd <- traj_mat[, -1, drop = FALSE] - traj_mat[, -ncol(traj_mat), drop = FALSE]
Xd <- sweep(Xd, 2, dt, "/")
Xd <- z_row(Xd)
stopifnot(nrow(X) == nrow(Xd)) # sanity


In [7]:
dim(X)
dim(Xd)

X = X[rownames(pc_df %>% filter(pvalue_1 < 0.05)), ]
Xd = Xd[rownames(pc_df %>% filter(pvalue_1 < 0.05)), ]

dim(X)
dim(Xd)


[1] 7513   97

[1] 7513   96

[1] 7121   97

[1] 7121   96

In [8]:
excl_genes.k562a <- read.csv(file.path(root_k562a, "notebooks", "exclude_time-tag-bias_genes.csv"))$X0
excl_genes.k562b <- read.csv(file.path(root_lps, "notebooks", "exclude_time-tag-bias_genes.k562.csv"))$X0
excl_genes.thp1 <- read.csv(file.path(root_lps, "notebooks", "exclude_time-tag-bias_genes.thp1.csv"))$X0
excl_genes.puro <- read.csv(file.path(root_puro, "notebooks", "exclude_time-tag-bias_genes.csv"))$X0

excl.genes <- c(
  rownames(pc_df %>% filter(cell_type == "K562a", gene %in% excl_genes.k562a)),
  rownames(pc_df %>% filter(cell_type == "K562b", gene %in% excl_genes.k562b)),
  rownames(pc_df %>% filter(cell_type == "THP1", gene %in% excl_genes.thp1)),
  rownames(pc_df %>% filter(cell_type == "K562_puro", gene %in% excl_genes.puro))
)
length(excl.genes)
head(excl.genes)

X <- X[!rownames(X) %in% excl.genes, ]
Xd <- Xd[!rownames(Xd) %in% excl.genes, ]

dim(X)
dim(Xd)


[1] 388

[1] "H2AC12_K562a"  "STMN1_K562a"   "NEAT1_K562a"   "EIF5A_K562a"  
[5] "HNRNPA1_K562a" "PRAME_K562a"

[1] 6749   97

[1] 6749   96

In [9]:
excl_genes.k562a <- read.csv(file.path(root_k562a, "notebooks", "exclude_low-count-genes.k562.csv"))$X0
excl_genes.k562b <- read.csv(file.path(root_lps, "notebooks", "exclude_low-count-genes.k562.csv"))$X0
excl_genes.thp1 <- read.csv(file.path(root_lps, "notebooks", "exclude_low-count-genes.thp1.csv"))$X0
excl_genes.puro <- read.csv(file.path(root_puro, "notebooks", "exclude_low-count-genes.k562.csv"))$X0

excl.genes <- c(
  rownames(pc_df %>% filter(cell_type == "K562a", gene %in% excl_genes.k562a)),
  rownames(pc_df %>% filter(cell_type == "K562b", gene %in% excl_genes.k562b)),
  rownames(pc_df %>% filter(cell_type == "THP1", gene %in% excl_genes.thp1)),
  rownames(pc_df %>% filter(cell_type == "K562_puro", gene %in% excl_genes.puro))
)
length(excl.genes)

X <- X[!rownames(X) %in% excl.genes, ]
Xd <- Xd[!rownames(Xd) %in% excl.genes, ]

dim(X)
dim(Xd)

X_mat <- as.matrix(X)
Xd_mat <- as.matrix(Xd)


[1] 32

[1] 6724   97

[1] 6724   96

In [10]:
pca_X <- prcomp(X_mat, center = TRUE, scale. = FALSE)
cumvar <- cumsum(pca_X$sdev^2) / sum(pca_X$sdev^2)
n_pc <- which(cumvar >= 0.99)[1] # retain enough PCs for 95% variance

X_pc <- pca_X$x[ , seq_len(n_pc), drop = FALSE]
n_pc


[1] 4

In [11]:
pca_Xd <- prcomp(Xd_mat, center = TRUE, scale. = FALSE)
cumvar <- cumsum(pca_Xd$sdev^2) / sum(pca_Xd$sdev^2)
n_pc <- which(cumvar >= 0.99)[1] # retain enough PCs for 95% variance

Xd_pc <- pca_Xd$x[ , seq_len(n_pc), drop = FALSE]
n_pc


[1] 5

In [12]:
pc_df = pc_df[rownames(pca_X$x), ]
pc_df$PC1 = pca_X$x[, 1]
pc_df$PC2 = pca_X$x[, 2]
pc_df$PC3 = pca_X$x[, 3]
pc_df$PC4 = pca_X$x[, 4]


In [13]:
# ============================================================
# 2. Plot clustering metrics
# ============================================================

plot_trajectory_clustering_metrics <- function(
  x,
  metrics_to_plot = c(
    "WSS",
    "p90_centroid_distance",
    "resolution",
    "mean_silhouette"
  ),
  overlay = FALSE,
  show_k = c("resolution", "elbow", "silhouette"),
  point_size = 1.8,
  line_width = 0.6
) {
  metrics <- x$metrics
  sel <- x$selection

  metric_labels <- c(
    WSS = "Within-cluster sum of squares",
    p90_centroid_distance = "90th percentile distance to centroid",
    resolution = "Fraction of achievable WSS improvement",
    mean_silhouette = "Mean silhouette"
  )

  df_long <- metrics %>%
    dplyr::select(K, dplyr::all_of(metrics_to_plot)) %>%
    tidyr::pivot_longer(-K, names_to = "metric", values_to = "value") %>%
    dplyr::mutate(
      metric = factor(
        metric,
        levels = metrics_to_plot,
        labels = metric_labels[metrics_to_plot]
      )
    )

  selection_df <- tibble::tibble(
    selection = c("resolution", "elbow", "silhouette"),
    K = c(sel$k_resolution, sel$k_elbow, sel$k_silhouette)
  ) %>%
    dplyr::filter(selection %in% show_k, !is.na(K)) %>%
    dplyr::mutate(
      selection = dplyr::recode(
        selection,
        resolution = paste0(round(100 * sel$wss_fraction), "% WSS resolution"),
        elbow = "Elbow",
        silhouette = "Maximum silhouette"
      )
    )

  if (!overlay) {
    return(
      ggplot2::ggplot(df_long, ggplot2::aes(K, value)) +
        ggplot2::geom_line(linewidth = line_width) +
        ggplot2::geom_point(size = point_size, shape = 1) +
        ggplot2::geom_vline(
          data = selection_df,
          ggplot2::aes(xintercept = K, linetype = selection),
          linewidth = 0.7
        ) +
        ggplot2::facet_wrap(~ metric, scales = "free_y", ncol = 1) +
        ggplot2::labs(x = "K", y = NULL, linetype = "K selection") +
        ggplot2::theme_bw() +
        ggplot2::theme(
          strip.background = ggplot2::element_blank(),
          strip.text = ggplot2::element_text(face = "bold"),
          legend.position = "top"
        )
    )
  }

  df_overlay <- df_long %>%
    dplyr::group_by(metric) %>%
    dplyr::mutate(
      value_scaled = {
        rng <- range(value, na.rm = TRUE)
        if (diff(rng) == 0) rep(0, dplyr::n()) else (value - rng[1]) / diff(rng)
      }
    ) %>%
    dplyr::ungroup()

  ggplot2::ggplot(
    df_overlay,
    ggplot2::aes(K, value_scaled, linetype = metric)
  ) +
    ggplot2::geom_line(linewidth = line_width) +
    ggplot2::geom_point(size = point_size, shape = 1) +
    ggplot2::geom_vline(
      data = selection_df,
      ggplot2::aes(xintercept = K),
      inherit.aes = FALSE,
      linewidth = 0.6,
      linetype = 2
    ) +
    ggplot2::labs(
      x = "K",
      y = "Metric scaled to [0,1]",
      linetype = "Metric"
    ) +
    ggplot2::theme_bw() +
    ggplot2::theme(legend.position = "top")
}


In [14]:
# ============================================================
# 1. Calculate clustering metrics across K
# ============================================================

calculate_trajectory_clustering_metrics <- function(
  X_mat,
  D,
  hc = NULL,
  kmeans_fits = NULL,
  K_grid = 3:200,
  wss_fraction = 0.95,
  select_k_by = c("resolution", "elbow", "silhouette")
) {
  select_k_by <- match.arg(select_k_by)

  X_mat <- as.matrix(X_mat)
  n <- nrow(X_mat)

  if (!inherits(D, "dist")) stop("D must be a 'dist' object.")
  if (attr(D, "Size") != n) stop("X_mat and D contain different numbers of observations.")

  # Provide either hierarchical clustering OR k-means fits
  if (is.null(hc) == is.null(kmeans_fits)) {
    stop("Provide exactly one of 'hc' or 'kmeans_fits'.")
  }

  if (!is.null(hc)) {
    if (!inherits(hc, "hclust")) stop("hc must be an 'hclust' object.")
    if (length(hc$order) != n) stop("X_mat and hc contain different numbers of observations.")
  }

  if (!is.null(kmeans_fits)) {
    if (is.null(names(kmeans_fits))) {
      stop("kmeans_fits must be named by K.")
    }

    # If K_grid wasn't supplied, use the fitted K values
    if (missing(K_grid)) {
      K_grid <- as.integer(names(kmeans_fits))
    }

    if (!all(as.character(K_grid) %in% names(kmeans_fits))) {
      stop("Some values in K_grid are not present in kmeans_fits.")
    }
  }

  if (max(K_grid) >= n) stop("max(K_grid) must be smaller than the number of genes.")

  calc_for_k <- function(K) {

    if (!is.null(hc)) {
      cl <- cutree(hc, k = K)

      centroids <- rowsum(X_mat, group = cl) / as.vector(table(cl))
      gene_centroids <- centroids[as.character(cl), , drop = FALSE]

    } else {
      fit <- kmeans_fits[[as.character(K)]]
      if (is.null(fit)) stop("kmeans_fits[['", K, "']] is NULL.")

      cl <- fit$cluster
      if (length(cl) != n) stop("K-means fit and X_mat contain different numbers of observations.")
      if (ncol(fit$centers) != ncol(X_mat)) stop("K-means fit and X_mat have different feature dimensions.")

      gene_centroids <- fit$centers[cl, , drop = FALSE]
    }

    d_to_centroid <- sqrt(rowSums((X_mat - gene_centroids)^2))
    sil <- cluster::silhouette(cl, D)

    data.frame(
      K = K,
      WSS = sum(d_to_centroid^2),
      p90_centroid_distance = unname(quantile(d_to_centroid, 0.90, na.rm = TRUE)),
      mean_silhouette = mean(sil[, "sil_width"], na.rm = TRUE)
    )
  }

  metrics <- dplyr::bind_rows(lapply(K_grid, calc_for_k))

  WSS_minK <- metrics$WSS[1]
  WSS_maxK <- metrics$WSS[nrow(metrics)]

  metrics <- metrics %>%
    dplyr::mutate(
      resolution = (WSS_minK - WSS) / (WSS_minK - WSS_maxK),
      k_norm = (K - min(K)) / (max(K) - min(K)),
      elbow_score = resolution - k_norm
    )

  eligible <- which(metrics$resolution >= wss_fraction)

  k_resolution <- if (length(eligible)) metrics$K[min(eligible)] else NA_integer_
  k_elbow <- metrics$K[which.max(metrics$elbow_score)]
  k_silhouette <- metrics$K[which.max(metrics$mean_silhouette)]

  k_selected <- switch(
    select_k_by,
    resolution = k_resolution,
    elbow = k_elbow,
    silhouette = k_silhouette
  )

  out <- list(
    metrics = metrics,
    selection = list(
      select_k_by = select_k_by,
      k_selected = k_selected,
      k_resolution = k_resolution,
      k_elbow = k_elbow,
      k_silhouette = k_silhouette,
      max_mean_silhouette = max(metrics$mean_silhouette, na.rm = TRUE),
      wss_fraction = wss_fraction
    )
  )

  class(out) <- c("trajectory_clustering_metrics", class(out))
  out
}


In [15]:
scale_by_total_column_variance <- function(M) {
  M <- as.matrix(M)
  scale_factor <- sqrt(sum(apply(M, 2, var)))
  if (!is.finite(scale_factor) || scale_factor == 0) {
    stop("Cannot scale matrix: total column variance is zero or non-finite.")
  }
  M / scale_factor
}

# Use direct trajectory and derivative matrices for clustering.
# Each block is scaled by total column variance before alpha weighting.
# PCA scores are retained for PC1-PC4 visualization only.
X_n <- scale_by_total_column_variance(X_mat)
Xd_n <- scale_by_total_column_variance(Xd_mat)

cl.ls <- list()
sil_df.ls <- list()
D.ls <- list()


In [16]:
lt = 0
alpha <- 0.25

subset.rownames = rownames(pc_df %>% filter(peak_log2fc > lt))
# subset.rownames = rownames(pc_df %>% filter(peak_log2fc < -lt))
# subset.rownames = rownames(pc_df)


In [17]:
X_joint <- cbind(sqrt(1 - alpha)*X_n, sqrt(alpha)*Xd_n)[subset.rownames, ]
D_joint <- dist(X_joint, method = "euclidean")

D_x <- dist(X_mat[subset.rownames, ], method = "euclidean") 
D_pca_x = dist(X_pc[subset.rownames, ], method = "euclidean")
D_pca_dx = dist(Xd_pc[subset.rownames, ], method = "euclidean")


In [18]:
D_use = D_joint
X_use = X_joint

hc <- hclust(D_use, method = "ward.D2")

clust_eval <- calculate_trajectory_clustering_metrics(X_mat = X_use, D = D_use, hc = hc, K_grid = 2:100, wss_fraction = 0.95, select_k_by = "elbow")
clust_eval$selection

options(repr.plot.width = 7.5, repr.plot.height = 5)
# plot_trajectory_clustering_metrics(clust_eval, overlay = FALSE)


$select_k_by
[1] "elbow"

$k_selected
[1] 15

$k_resolution
[1] 50

$k_elbow
[1] 15

$k_silhouette
[1] 3

$max_mean_silhouette
[1] 0.4772855

$wss_fraction
[1] 0.95

In [19]:
cl <- cutree(hc, k = clust_eval$selection$k_silhouette)
gene_cluster_distance <- rep(NA_real_, nrow(X_use))
names(gene_cluster_distance) <- rownames(X_use)

for (k in sort(unique(cl))) {
  idx <- which(cl == k)
  Xi <- X_use[idx, , drop = FALSE]
  centroid <- colMeans(Xi)
  gene_cluster_distance[idx] <- sqrt(rowSums(sweep(Xi, 2, centroid, "-")^2))
}

disp_df <- data.frame(gene = rownames(X_use), cluster = cl, centroid_dist = gene_cluster_distance, stringsAsFactors = FALSE)

name <- "broad_induced"
cl.ls[[name]] <- cl
sil_df.ls[[name]] <- disp_df

cl <- cutree(hc, k = clust_eval$selection$k_elbow)
gene_cluster_distance <- rep(NA_real_, nrow(X_use))
names(gene_cluster_distance) <- rownames(X_use)

for (k in sort(unique(cl))) {
  idx <- which(cl == k)
  Xi <- X_use[idx, , drop = FALSE]
  centroid <- colMeans(Xi)
  gene_cluster_distance[idx] <- sqrt(rowSums(sweep(Xi, 2, centroid, "-")^2))
}

disp_df <- data.frame(gene = rownames(X_use), cluster = cl, centroid_dist = gene_cluster_distance, stringsAsFactors = FALSE)

name <- "fine_induced"
cl.ls[[name]] <- cl
sil_df.ls[[name]] <- disp_df

for (name in c("broad_induced", "fine_induced")) {
    # Add assignments back into pc_df
    pc_df[[paste0("cluster_", name)]] <- as.factor(cl.ls[[name]][rownames(pc_df)])
    pc_df[[paste0("cluster_dist_", name)]] <- sil_df.ls[[name]]$centroid_dist[match(rownames(pc_df), sil_df.ls[[name]]$gene)]
    
    # inspect cluster sizes
    print(name)
    print(sort(table(cl.ls[[name]]), decreasing = TRUE))
    
    cluster_name = paste0("cluster_", name)
    pc_df[[cluster_name]] <- factor(pc_df[[cluster_name]], levels = c(levels(pc_df[[cluster_name]]), "-1"))
    pc_df[[cluster_name]][is.na(pc_df[[cluster_name]])] <- "-1"
    pc_df[[cluster_name]] <- factor(pc_df[[cluster_name]])
}


[1] "broad_induced"

   2    1    3 
1846  742  623 
[1] "fine_induced"

  2   7   5  13   6   4   3   8  11  10   1   9  12  15  14 
968 613 239 208 204 189 170 165 126  85  72  57  43  43  29 


In [20]:
subset.rownames = rownames(pc_df %>% filter(peak_log2fc < -lt))

X_joint <- cbind(sqrt(1 - alpha)*X_n, sqrt(alpha)*Xd_n)[subset.rownames, ]
D_joint <- dist(X_joint, method = "euclidean")

D_x <- dist(X_mat[subset.rownames, ], method = "euclidean") 
D_pca_x = dist(X_pc[subset.rownames, ], method = "euclidean")
D_pca_dx = dist(Xd_pc[subset.rownames, ], method = "euclidean")

D_use = D_joint
X_use = X_joint

hc <- hclust(D_use, method = "ward.D2")

clust_eval <- calculate_trajectory_clustering_metrics(X_mat = X_use, D = D_use, hc = hc, K_grid = 2:100, wss_fraction = 0.95, select_k_by = "elbow")
clust_eval$selection

options(repr.plot.width = 7.5, repr.plot.height = 5)
# plot_trajectory_clustering_metrics(clust_eval, overlay = FALSE)


$select_k_by
[1] "elbow"

$k_selected
[1] 18

$k_resolution
[1] 52

$k_elbow
[1] 18

$k_silhouette
[1] 3

$max_mean_silhouette
[1] 0.5472789

$wss_fraction
[1] 0.95

In [21]:
cl <- cutree(hc, k = clust_eval$selection$k_silhouette)
gene_cluster_distance <- rep(NA_real_, nrow(X_use))
names(gene_cluster_distance) <- rownames(X_use)

for (k in sort(unique(cl))) {
  idx <- which(cl == k)
  Xi <- X_use[idx, , drop = FALSE]
  centroid <- colMeans(Xi)
  gene_cluster_distance[idx] <- sqrt(rowSums(sweep(Xi, 2, centroid, "-")^2))
}

disp_df <- data.frame(gene = rownames(X_use), cluster = cl, centroid_dist = gene_cluster_distance, stringsAsFactors = FALSE)

name <- "broad_repressed"
cl.ls[[name]] <- cl
sil_df.ls[[name]] <- disp_df

cl <- cutree(hc, k = clust_eval$selection$k_elbow)
gene_cluster_distance <- rep(NA_real_, nrow(X_use))
names(gene_cluster_distance) <- rownames(X_use)

for (k in sort(unique(cl))) {
  idx <- which(cl == k)
  Xi <- X_use[idx, , drop = FALSE]
  centroid <- colMeans(Xi)
  gene_cluster_distance[idx] <- sqrt(rowSums(sweep(Xi, 2, centroid, "-")^2))
}

disp_df <- data.frame(gene = rownames(X_use), cluster = cl, centroid_dist = gene_cluster_distance, stringsAsFactors = FALSE)

name <- "fine_repressed"
cl.ls[[name]] <- cl
sil_df.ls[[name]] <- disp_df

for (name in c("broad_repressed", "fine_repressed")) {
    # Add assignments back into pc_df
    pc_df[[paste0("cluster_", name)]] <- as.factor(cl.ls[[name]][rownames(pc_df)])
    pc_df[[paste0("cluster_dist_", name)]] <- sil_df.ls[[name]]$centroid_dist[match(rownames(pc_df), sil_df.ls[[name]]$gene)]
    
    # inspect cluster sizes
    print(name)
    print(sort(table(cl.ls[[name]]), decreasing = TRUE))
    
    cluster_name = paste0("cluster_", name)
    pc_df[[cluster_name]] <- factor(pc_df[[cluster_name]], levels = c(levels(pc_df[[cluster_name]]), "-1"))
    pc_df[[cluster_name]][is.na(pc_df[[cluster_name]])] <- "-1"
    pc_df[[cluster_name]] <- factor(pc_df[[cluster_name]])
}


[1] "broad_repressed"

   1    2    3 
1993 1210  310 
[1] "fine_repressed"

   1    5    4    3    6   11   12   13    9    8   17   16   14   10    2   18 
1360  356  275  242  226  170  146  137  114  109   84   75   53   48   32   32 
   7   15 
  27   27 


In [22]:
RenameClusters <- function(x) {
    x <- as.character(x)
    if (x == '1') { return('impulse-up_1') }
    if (x == '2') { return('trend-up') }
    if (x == '3') { return('impulse-up_2') }
    else { return(x) }
}

pc_df$cluster_broad_induced = sapply(pc_df$cluster_broad_induced, RenameClusters, USE.NAMES = F)


In [23]:
RenameClusters <- function(x) {
  x <- as.character(x)
  if (x == "1") return("trend-down_1")
  if (x == "2") return("trend-down_2")
  if (x == "3") return("transient-down")
  x
}

pc_df$cluster_broad_repressed <- sapply(pc_df$cluster_broad_repressed, RenameClusters, USE.NAMES = FALSE)


In [24]:
pc_df$broad_cluster <- factor(ifelse(pc_df$cluster_broad_induced != "-1", as.character(pc_df$cluster_broad_induced), as.character(pc_df$cluster_broad_repressed)))

table(pc_df$broad_cluster)



  impulse-up_1   impulse-up_2 transient-down   trend-down_1   trend-down_2 
           742            623            310           1993           1210 
      trend-up 
          1846 

In [25]:
pc_df$cluster_dist_broad <- dplyr::coalesce(pc_df$cluster_dist_broad_induced, pc_df$cluster_dist_broad_repressed)
pc_df$cluster_dist_fine <- dplyr::coalesce(pc_df$cluster_dist_fine_induced, pc_df$cluster_dist_fine_repressed)


In [26]:
write.csv(pc_df, file.path(SELF_CONTAINED_OUTDIR, "pc_df.new.csv"))
saveRDS(pc_df, file.path(SELF_CONTAINED_OUTDIR, "pc_df.new.rds"))

cat("Wrote:\n")
cat("  ", file.path(SELF_CONTAINED_OUTDIR, "pc_df.new.csv"), "\n", sep = "")
cat("  ", file.path(SELF_CONTAINED_OUTDIR, "pc_df.new.rds"), "\n", sep = "")


Wrote:


  /new-stg/home/alex/tmp/ChronoPack/k562-tnfa-puromycin.hg38/self_contained_outputs/global_analysis_extended/pc_df.new.csv


  /new-stg/home/alex/tmp/ChronoPack/k562-tnfa-puromycin.hg38/self_contained_outputs/global_analysis_extended/pc_df.new.rds


In [27]:
VALIDATE_AGAINST_REFERENCE <- identical(Sys.getenv("CHRONOPACK_VALIDATE_REFERENCE"), "1")
if (VALIDATE_AGAINST_REFERENCE) {
  reference_path <- file.path(root_puro, "intmd", "global_analysis", "extended", "pc_df.new.rds")
  ref <- readRDS(reference_path)
  common_rows <- intersect(rownames(ref), rownames(pc_df))
  common_cols <- intersect(colnames(ref), colnames(pc_df))
  cat("Reference rows:", nrow(ref), " regenerated rows:", nrow(pc_df), " common:", length(common_rows), "\n")
  cat("Common columns:", length(common_cols), "\n")
  for (nm in intersect(c("broad_cluster", "cluster_broad_induced", "cluster_broad_repressed", "cluster_fine_induced", "cluster_fine_repressed"), common_cols)) {
    cat(nm, "match fraction:", mean(as.character(ref[common_rows, nm]) == as.character(pc_df[common_rows, nm]), na.rm = TRUE), "\n")
  }
  numeric_cols <- common_cols[vapply(pc_df[common_cols], is.numeric, logical(1)) & vapply(ref[common_cols], is.numeric, logical(1))]
  max_abs <- vapply(numeric_cols, function(nm) max(abs(ref[common_rows, nm] - pc_df[common_rows, nm]), na.rm = TRUE), numeric(1))
  print(sort(max_abs, decreasing = TRUE))
}


Reference rows: 6724  regenerated rows: 6724  common: 6724 
Common columns: 35 
broad_cluster match fraction: 1 
cluster_broad_induced match fraction: 1 
cluster_broad_repressed match fraction: 1 
cluster_fine_induced match fraction: 1 
cluster_fine_repressed match fraction: 1 
                   peak_dexp                     max_curv 
                5.235111e+01                 1.541443e-08 
                         PC1                          PC4 
                2.248597e-09                 1.688208e-09 
                         PC2                    mean_curv 
                6.083924e-10                 1.876308e-10 
                         PC3                  peak_log2fc 
                1.830891e-10                 9.585299e-11 
                     owp_ddt                     auc_norm 
                9.047502e-11                 8.387008e-11 
                         owp   cluster_dist_broad_induced 
                8.089840e-11                 7.715850e-11 
          clu